# Scaling with a better prior (100-wire circuit, low data)

The question this notebook attacks: **does the generalization recipe from
`FINDINGS_iter.md` (transient init-relative weight noise + adam_eps 1e-4)
change the model-size scaling curve, or just shift it?** The tiny 8-wire
task had no headroom — the smallest model already hit the Bayes ceiling —
so this moves to a task family with a real difficulty gradient:

- **100 wires, circuit depth 8**, uniform tap depths 1-8: all 100 outputs
  supervised jointly, so hardness ranges from trivial (depth 1) to deep
  (depth 8) within one run, and "scale" shows up as *how deep a model can
  solve*, not a single saturating number.
- **`train_n=2048`**: a fixed pool of 2048 sampled inputs (the 2^100 input
  space is not enumerable — pool and held-out sets are disjoint w.h.p.).
  At batch 256 that is 8 steps per epoch: heavy repetition, the low-data
  regime. Held-out eval uses 2000 fresh inputs.
- 7 shapes x {clean, recipe}, 1 seed each, 10k steps: ~30-45 min on a T4.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import os
    %pip -q install -U "jax[cuda12]" optax
    if not os.path.exists("/content/circscale"):
        !git clone https://github.com/amdson/circscale.git /content/circscale
    %cd /content/circscale
    !git pull
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/circscale_runs", exist_ok=True)
    if not os.path.islink("runs"):
        os.symlink("/content/drive/MyDrive/circscale_runs", "runs")

## Config

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from train import RunConfig, load_run, run

N_WIRES, CIRC_DEPTH, CIRCUIT_SEED = 100, 8, 0
TRAIN_N = 2048
STEPS = 10_000
SEEDS = [0]
OUT_DIR = "runs/scale100"
CHANCE = float(np.log(2))

# (width, mlp_depth, lr) — tuned Adam LRs (runs/lr_table.json)
SHAPES = [
    (32,  2, 1e-2),
    (64,  3, 1e-2),
    (128, 4, 3e-3),
    (180, 5, 3e-3),
    (256, 6, 1e-3),
    (360, 7, 1e-3),
    (512, 8, 1e-3),
]
# arm name -> RunConfig overrides (recipe = FINDINGS_iter.md)
ARMS = {
    "clean":  dict(),
    "recipe": dict(weight_noise=0.5, adam_eps=1e-4),
}


def cfg100(width, depth, lr, seed=0, **kw):
    return RunConfig(width=width, mlp_depth=depth, lr=lr, steps=STEPS,
                     n_wires=N_WIRES, circ_depth=CIRC_DEPTH,
                     circuit_seed=CIRCUIT_SEED, train_n=TRAIN_N,
                     eval_every=200, eval_n=2000, model_seed=seed,
                     out_dir=OUT_DIR, **kw)


def n_params(w, d, hr=4):
    h = hr * w
    return N_WIRES * w + d * (w + w * h + h * w) + w + w * N_WIRES


print(f"{len(SHAPES)} shapes x {len(ARMS)} arms x {len(SEEDS)} seeds, "
      f"{STEPS:,} steps, pool {TRAIN_N} (8 steps/epoch at batch 256)")

## Run (idempotent — interrupt and re-run freely)

In [ ]:
res = {}
for w, d, lr in SHAPES:
    for arm, over in ARMS.items():
        for s in SEEDS:
            cfg = cfg100(w, d, lr, seed=s, **over)
            run(cfg)
            res[(w, d, arm, s)] = load_run(cfg.npz_path)[1]
print(f"{len(res)} runs done")

## Held-out accuracy by tap depth

The scaling question in one figure: each panel is an arm; lines are shapes
(dark = small, light = large); x is circuit tap depth. A model "reaches"
depth k if its held-out accuracy on depth-k wires is high. If the recipe
only shifts curves up, it is a better prior at fixed capability; if it lets
larger models reach *deeper* than clean training does, compute-for-prior
bends the scaling curve.

In [ ]:
depths = res[(SHAPES[0][0], SHAPES[0][1], "clean", SEEDS[0])]["out_depths"]
dvals = np.arange(1, CIRC_DEPTH + 1)
cols = plt.cm.viridis(np.linspace(0, 0.9, len(SHAPES)))

fig, axes = plt.subplots(1, len(ARMS), figsize=(6 * len(ARMS), 4.4),
                         sharey=True, squeeze=False)
for ax, arm in zip(axes[0], ARMS):
    for (w, d, lr), col in zip(SHAPES, cols):
        accs = np.mean([res[(w, d, arm, s)]["per_out_acc_ho"][-1]
                        for s in SEEDS], axis=0)
        ax.plot(dvals, [accs[depths == k].mean() for k in dvals],
                "-o", color=col, ms=4, label=f"w{w}d{d}")
    ax.axhline(0.5, color="gray", ls=":", lw=0.8)
    ax.set(xlabel="tap depth", title=arm, ylim=(0.45, 1.02))
axes[0, 0].set(ylabel="final held-out acc (mean over wires at depth)")
axes[0, 0].legend(fontsize=7)
plt.tight_layout()

## Scaling curves

Left: wires solved (held-out acc > 0.95) vs params. Right: mean held-out
BCE vs params, log-log — the L(N) curve, clean vs recipe.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.4))
for arm, col in zip(ARMS, ("C0", "C1")):
    Ns, solved, bce = [], [], []
    for w, d, lr in SHAPES:
        acc = np.mean([res[(w, d, arm, s)]["per_out_acc_ho"][-1]
                       for s in SEEDS], axis=0)
        loss = np.mean([res[(w, d, arm, s)]["per_out_loss_ho"][-1]
                        for s in SEEDS], axis=0)
        Ns.append(n_params(w, d))
        solved.append((acc > 0.95).sum())
        bce.append(loss.mean())
    ax1.plot(Ns, solved, "-o", color=col, label=arm)
    ax2.plot(Ns, bce, "-o", color=col, label=arm)
ax1.set(xscale="log", xlabel="params N", ylabel=f"wires solved / {N_WIRES}")
ax1.legend(fontsize=8)
ax2.axhline(CHANCE, color="gray", ls=":", lw=0.8)
ax2.set(xscale="log", yscale="log", xlabel="params N",
        ylabel="mean held-out BCE")
ax2.legend(fontsize=8)
plt.tight_layout()

## Train vs held-out over training (w128d4 and w512d8)

Generalization-gap dynamics per arm: train-pool (dashed) vs held-out
(solid) mean BCE.

In [ ]:
pick = [(128, 4), (512, 8)]
fig, axes = plt.subplots(1, len(pick), figsize=(6 * len(pick), 4.4),
                         sharey=True, squeeze=False)
for ax, (w, d) in zip(axes[0], pick):
    for arm, col in zip(ARMS, ("C0", "C1")):
        r = res[(w, d, arm, SEEDS[0])]
        m = r["eval_steps"] > 0
        ax.plot(r["eval_steps"][m], r["per_out_loss_tr"][m].mean(axis=1),
                "--", color=col, lw=1.1)
        ax.plot(r["eval_steps"][m], r["per_out_loss_ho"][m].mean(axis=1),
                "-", color=col, lw=1.3, label=arm)
    ax.axhline(CHANCE, color="gray", ls=":", lw=0.8)
    ax.set(xscale="log", yscale="log", xlabel="step", title=f"w{w}d{d}")
axes[0, 0].set(ylabel="mean BCE (dashed=train pool, solid=held-out)")
axes[0, 0].legend(fontsize=8)
plt.tight_layout()

## Notes

- `train_n` pools are sampled, not enumerated: `per_out_loss`/`_ho` are the
  2000 fresh held-out inputs, `_tr` is the 2048-input pool. Disjoint w.h.p.
  at n=100.
- Tap depths are uniform on [1, 8]; with 100 wires that is ~12 wires per
  depth, so per-depth means average ~12 wires.
- One seed per cell to keep this small; add seeds to `SEEDS` for any cell
  that looks pivotal (runs are idempotent). The recipe arm uses the tuned
  per-shape LRs, not the findings' lr~1/width width recipe — if wide
  recipe runs underperform, try `weight_decay` + `wd_scale="init"` per
  FINDINGS_iter.md before concluding scale doesn't help.
- Deeper-tap wires may need more than 10k steps; treat "solved at 10k" as
  a lower bound on capability.